worked on by:

Building models

The previous part ended with a single notebook (per dataset) that would prepare your data for predicting. Next is building a couple of models and actually predicting something.

Which models will you need?

    A quick first model. This won't be a good one, but with this you can start working on the deployment (next step) while still tuning the model.
    Use PyCaret (or another automated ML comparison) on both datasets.
    Create and tune a model on both your datasets. Explain why you choose this particular model and perhaps train a second model to validate this choice.
    Create and tune a model on AWS.

Make sure to keep all the metrics on the models you made and compare these to show which model performed best.

# Predict

In [1]:
%pip install pycaret[full]

Note: you may need to restart the kernel to use updated packages.Collecting pycaret[full]

  Using cached pycaret-3.3.2-py3-none-any.whl (486 kB)
  Using cached imbalanced_learn-0.14.0-py3-none-any.whl (239 kB)
  Using cached yellowbrick-1.5-py3-none-any.whl (282 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
  Using cached scikit_plot-0.3.7-py3-none-any.whl (33 kB)
  Using cached plotly-6.5.0-py3-none-any.whl (9.9 MB)
  Using cached sktime-0.26.0-py3-none-any.whl (21.8 MB)
  Using cached joblib-1.3.2-py3-none-any.whl (302 kB)
  Using cached schemdraw-0.15-py3-none-any.whl (106 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
  Using cached pyod-2.0.5-py3-none-any.whl (200 kB)
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
  Using cached pandas-2.1.4-cp310-cp310-win_amd64.whl (10.7 MB)
  Using cached kaleido-1.2.0-py3-none-any.whl (68 kB)
  Using cached statsmodels-0.14.5-cp3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.25.1 requires huggingface-hub<1.0,>=0.10.0, but you have huggingface-hub 1.1.5 which is incompatible.
open-clip-torch 2.7.0 requires protobuf==3.20.0, but you have protobuf 6.33.1 which is incompatible.
omegaconf 2.2.3 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.

[notice] A new release of pip available: 22.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from pycaret.regression import setup, compare_models, evaluate_model, finalize_model, save_model

In [3]:
df = pd.read_csv("cleaned_price_paid_records.csv", parse_dates=["date_of_transfer"])

df["year"] = df["date_of_transfer"].dt.year
df["month"] = df["date_of_transfer"].dt.month
df = df.drop(columns=["date_of_transfer"])

reg_setup = setup(
    data=df.sample(50000, random_state=42),
    target="price",
    session_id=42,
    categorical_features=[
        "property_type",
        "old_or_new",
        "duration",
        "town_or_city",
        "district",
        "county",
    ],
    normalize=True,
    verbose=True,
)

best_model = compare_models()
best_model
save_model(best_model, "house_price_pycaret_model")

,Description,Value
0,Session id,42
1,Target,price
2,Target type,Regression
3,Original data shape,"(50000, 9)"
4,Transformed data shape,"(50000, 12)"
5,Transformed train set shape,"(35000, 12)"
6,Transformed test set shape,"(15000, 12)"
7,Numeric features,2
8,Categorical features,6
9,Preprocess,True


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,58713.3615,30639489165.2047,167046.0806,0.4506,0.4310,0.4089,0.4540
gbr,Gradient Boosting Regressor,60121.0121,33121210734.4225,173269.3840,0.4109,0.4429,0.4119,0.5050
catboost,CatBoost Regressor,57457.7303,32954515775.6320,171622.9256,0.4106,0.4197,0.3857,3.9880
rf,Random Forest Regressor,61750.7205,34030106940.4988,175739.3170,0.3755,0.4430,0.4332,1.8280
xgboost,Extreme Gradient Boosting,58834.3230,33513878323.2000,175078.3258,0.3746,0.4265,0.3971,0.5200
knn,K Neighbors Regressor,64227.4262,34514160742.4000,178165.7727,0.3700,0.4642,0.4569,0.1800
lar,Least Angle Regression,75400.5304,37433682681.1056,183769.4534,0.3418,0.7369,0.6342,0.0810
llar,Lasso Least Angle Regression,75399.7477,37433651023.1067,183769.2952,0.3418,0.7368,0.6342,0.0740
br,Bayesian Ridge,75373.2263,37432592740.5751,183763.5615,0.3418,0.7373,0.6339,0.0760
ridge,Ridge Regression,75399.3961,37433645251.3855,183769.2244,0.3418,0.7369,0.6342,0.0800


Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['year', 'month'],
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=['property_type', 'old_or_new',
                                              'duration', 'town_or_city',
                                              'district', 'county'],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('ordinal_encoding',
                  Transform...
                                                               handle_missing='return_nan',
                                                               use_cat_names=True))),
                 ('rest_encoding',
                  TransformerWrapper(include=['town_or_city', 'district',
                                              'county'],
                       